In [24]:
!pip install pyomo
!pip install cbcbox

import pyomo.environ as pyo

model = pyo.ConcreteModel()

model.cand = pyo.Set(initialize=['kc','dh','hb','sc','ks','nk'])
model.day = pyo.Set(initialize=['m', 't', 'w', 'th' ,'f'])

avail = {('kc','m'):6, ('kc','t'):0, ('kc','w'):6, ('kc','th'):0, ('kc','f'):6, 
         ('dh','m'):0, ('dh','t'):6, ('dh','w'):0, ('dh','th'):6, ('dh','f'):0,
         ('hb','m'):4, ('hb','t'):8, ('hb','w'):4, ('hb','th'):0, ('hb','f'):4, 
         ('sc','m'):5, ('sc','t'):5, ('sc','w'):5, ('sc','th'):0, ('sc','f'):5,
         ('ks','m'):3, ('ks','t'):0, ('ks','w'):3, ('ks','th'):8, ('ks','f'):0, 
         ('nk','m'):0, ('nk','t'):0, ('nk','w'):0, ('nk','th'):6, ('nk','f'):2}

wage = {'kc':150, 
        'dh':152,
        'hb':148, 
        'sc':146,
        'ks':166,
        'nk':176}

min_work = {'kc':8, 
        'dh':8,
        'hb':8, 
        'sc':8,
        'ks':7,
        'nk':7}

model.avail = pyo.Param(model.cand, model.day, initialize=avail)
model.wage = pyo.Param(model.cand, initialize=wage)
model.min_work = pyo.Param(model.cand, initialize=min_work)


model.x = pyo.Var(model.cand, model.day, within=pyo.NonNegativeReals)


def total_cost(model):
    return sum(model.wage[c]*model.x[c, d] for d in model.day for c in model.cand)

model.objective = pyo.Objective(rule=total_cost, sense=pyo.minimize)


def contract_min(model, c):
    return sum(model.x[c, d] for d in model.day) >= model.min_work[c]

model.contract_min = pyo.Constraint(model.cand, rule=contract_min)


def avail_con(model, c, d):
    return model.x[c, d] <= model.avail[c, d]

model.avail_con = pyo.Constraint(model.cand, model.day, rule=avail_con)


def demand(model, d):
    return sum(model.x[c, d] for c in model.cand) >= 14

model.demand = pyo.Constraint(model.day, rule=demand)


solver = pyo.SolverFactory('cbc')
results = solver.solve(model, tee=True)
model.x.pprint()

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
CBC devel (git:146ce89) -- COIN-OR Branch and Cut
  args: -printingOptions all -import /tmp/tmpsi_0f7uv.pyomo.lp -stat 1 -solve -solu /tmp/tmpsi_0f7uv.pyomo.soln
  default strategy 1

> Problem loading

  |Rows| = 41   |Cols| = 30   |NZ| = 90
  Variables: 30 continuous

  Coefficient ranges:
    Matrix  in [  1,   1]   ratio =      1
    Cost    in [146, 176]   ratio = 1.2055
    Bounds  in [  -,   -]   ratio =      -
    RHS     in [  2,  14]   ratio =      7

+ Problem loaded - 41 rows, 30 cols, 90 NZ
Statistics for presolved model


Problem has 11 rows, 18 columns (18 with objective) and 36 elements
Column breakdown:
0 of type 0.0->inf, 18 of type 0.0->up, 0 of type lo->inf, 
0 of type lo->up, 0 of type free, 0 of type fixed, 
0 of type -inf->0.0, 0 of type -inf->up, 0 of type 0.0->1.0 
Row breakdown:
0 of type E 0.0, 0 of type E